# Dynamo Disaggregated Inference — Results Analysis

This notebook loads benchmark JSON files from `$RESULTS_DIR`, computes summary statistics,
and generates comparison plots across disaggregation configs.

**Configs compared:**
- `agg_baseline` — standard aggregated vLLM (no disaggregation)
- `disagg_1p1d` — 1 prefill + 1 decode
- `disagg_2p1d` — 2 prefill + 1 decode
- `disagg_2p2d` — 2 prefill + 2 decode

**Metrics:** TTFT, TPOT, ITL, E2EL at p50 / p99; request throughput.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

SCRATCH = '/storage/ice1/4/5/ealbalas3/disaggregated-prefill-decode-research'
RESULTS_DIR = Path(SCRATCH) / 'results/dynamo'
PLOTS_DIR   = RESULTS_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Results dir: {RESULTS_DIR}')
json_files = sorted(RESULTS_DIR.glob('*.json'))
print(f'Found {len(json_files)} result files.')

In [ ]:
METRIC_KEYS = [
    'mean_ttft_ms', 'median_ttft_ms', 'p99_ttft_ms',
    'mean_tpot_ms', 'median_tpot_ms', 'p99_tpot_ms',
    'mean_itl_ms',  'median_itl_ms',  'p99_itl_ms',
    'mean_e2el_ms', 'median_e2el_ms', 'p99_e2el_ms',
    'request_throughput', 'output_throughput',
]

def config_label(stem: str) -> str:
    """Strip timestamp / job ID suffix to get the config name."""
    parts = stem.split('_')
    label_parts = []
    for p in parts:
        if (p.startswith('job') and p[3:].isdigit()) or (len(p) == 8 and p.isdigit()):
            break
        label_parts.append(p)
    return '_'.join(label_parts)

records = []
for path in json_files:
    with open(path) as f:
        data = json.load(f)
    rec = {'file': path.name, 'tag': path.stem, 'config': config_label(path.stem)}
    stem = path.stem
    for part in stem.split('_'):
        if part.startswith('in') and part[2:].isdigit():
            rec['input_len'] = int(part[2:])
        elif part.startswith('out') and part[3:].isdigit():
            rec['output_len'] = int(part[3:])
        elif part.startswith('rate') and part[4:].replace('.','').isdigit():
            rec['rate'] = float(part[4:])
    for k in METRIC_KEYS:
        rec[k] = data.get(k, float('nan'))
    records.append(rec)

df = pd.DataFrame(records)
print(df[['config', 'input_len', 'output_len', 'rate', 'mean_ttft_ms', 'mean_tpot_ms', 'p99_e2el_ms', 'request_throughput']].to_string(index=False))

In [ ]:
# Summary: mean across all runs per config
summary = (
    df.groupby('config')[METRIC_KEYS]
    .mean()
    .reset_index()
    .sort_values('config')
)
summary

In [ ]:
def bar_comparison(summary_df, metric, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(8, 5))
    configs = summary_df['config'].tolist()
    values  = summary_df[metric].tolist()
    x = np.arange(len(configs))
    bars = ax.bar(x, values, color='steelblue', edgecolor='white')
    ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(configs, rotation=20, ha='right')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    path = PLOTS_DIR / filename
    fig.savefig(path, dpi=150)
    plt.show()
    print(f'Saved: {path}')

bar_comparison(summary, 'mean_ttft_ms',       'Mean TTFT (ms)',       'Mean TTFT by Config',          'ttft_mean.png')
bar_comparison(summary, 'p99_ttft_ms',        'p99 TTFT (ms)',        'p99 TTFT by Config',           'ttft_p99.png')
bar_comparison(summary, 'mean_tpot_ms',       'Mean TPOT (ms)',       'Mean TPOT by Config',          'tpot_mean.png')
bar_comparison(summary, 'p99_e2el_ms',        'p99 E2EL (ms)',        'p99 End-to-End Latency',       'e2el_p99.png')
bar_comparison(summary, 'request_throughput', 'Request Throughput (req/s)', 'Throughput by Config',  'throughput.png')

In [ ]:
# TTFT vs Throughput scatter — one point per result file, colored by config
if len(df) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    configs_list = df['config'].unique().tolist()
    cmap = plt.get_cmap('tab10')
    for i, cfg in enumerate(sorted(configs_list)):
        sub = df[df['config'] == cfg]
        ax.scatter(
            sub['request_throughput'],
            sub['mean_ttft_ms'],
            label=cfg,
            color=cmap(i),
            s=80, zorder=3,
        )
    ax.set_xlabel('Request Throughput (req/s)')
    ax.set_ylabel('Mean TTFT (ms)')
    ax.set_title('TTFT vs Throughput — Disagg vs Baseline')
    ax.legend()
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    path = PLOTS_DIR / 'ttft_vs_throughput.png'
    fig.savefig(path, dpi=150)
    plt.show()
    print(f'Saved: {path}')

In [ ]:
# Improvement over baseline (agg_baseline = 1.0×)
if 'agg_baseline' in summary['config'].values:
    baseline = summary[summary['config'] == 'agg_baseline'].iloc[0]
    improvement = summary.copy()
    for m in ['mean_ttft_ms', 'p99_ttft_ms', 'mean_tpot_ms', 'p99_e2el_ms']:
        improvement[f'{m}_ratio'] = improvement[m] / baseline[m]
    improvement[['config', 'mean_ttft_ms_ratio', 'p99_ttft_ms_ratio',
                 'mean_tpot_ms_ratio', 'p99_e2el_ms_ratio']]
else:
    print('No agg_baseline results found yet — run job_agg.sh first.')